In [1]:
import torch
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

## Input and ground truth initialization

In [2]:
def InputInit(batch_size, image_height, image_width, num_keypoints):
  '''
  Generates data to represent the random input
  the input should be: (batch_size, num_keypoints, img_height, img_width),
  and we set the num_keypoints in the image with 1
  '''
  input = torch.rand(batch_size, num_keypoints, image_height, image_width)
  return input


In [3]:
def GroundTruthInit(batch_size, image_height, image_width, num_keypoints):
  '''
  Generate data to represent the ground truth "label"
  The label shape should be: (batch_size, num_keypoints, img_height, img_width)
  '''
  ground_truth = torch.rand(batch_size, num_keypoints, image_height, image_width)
  return ground_truth

## Compute the main loss function

In [4]:
def LossFunction(predicted_output, ground_truth):
  '''
  Calculate the L2_Loss between the model output and the ground truth
  where if it's just one image it goes with the second condition, but if
  the batch size is more than one, it goes with the first condition.
  '''
  if((predicted_output.shape[0])>1):
    squared_diff = (predicted_output - ground_truth)**2 # comput the squared difference
    l2_loss = torch.mean(squared_diff) # find the mean
    return l2_loss

  else:
    #print('less than 2d' )
    l2_loss = F.mse_loss(predicted_output, ground_truth) # calculate mean square error loss
  return l2_loss

**Anatomical-context weight** (from the task spec, eq. 6):

$$\omega_{ac}(h^c, o^c) = \log_2 \lVert A(h^c) - A(o^c) \rVert_2^2 + \log_2 \lVert D(h^c) - D(o^c) \rVert_2^2$$

where $A(\cdot)$ is the matrix of angles and $D(\cdot)$ the matrix of distances between every pair of landmarks, for the ground-truth heatmaps $h^c$ and the predicted heatmaps $o^c$.

## Compute weight Paramater

In [5]:
def GlobalMaxDistance(image):
  '''
  Calculate the euclidean distances matrix between each pair of the points; where the extracted
  points are the x,y of the maximum pixel
  '''
  reshaped_image = torch.reshape(image, (image.size(0), image.size(1), -1)) # reshape the image to get the max value and its indicies

  max_values, max_indices = torch.max(reshaped_image, dim=2) # Find the maximum pixel values and their indices

  # Calculate the x and y indices from the flattened index
  width = image.size(3)
  x_indices = max_indices // width
  x_indices = torch.tensor(x_indices, dtype=torch.float)
  y_indices = max_indices % width
  y_indices = torch.tensor(y_indices, dtype = torch.float)

  points = torch.stack((x_indices.squeeze(), y_indices.squeeze()), dim=-1) # reshpae the points into(-1, no.channels,2)
  distances = torch.cdist(points, points) # calculate l2 distance between each pair of the points

  return  distances

In [6]:
def GlobalMaxAngle(image):
  '''
  Calculate the radiant angels matrix between each pair of the points; where the extracted
  points are the x,y of the maximum pixel.

  if the points is for just one image (one batch) it goes for the second condition,
  and if the points are for more one image (more than one batch) it goes for the first condition.
  '''

  reshaped_image = torch.reshape(image, (image.size(0), image.size(1), -1)) # reshape the image to get the max value and its indicies

  max_values, max_indices = torch.max(reshaped_image, dim=2) ## Find the maximum pixel values and their indices
  # Calculate the x and y indices from the flattened index
  width = image.size(3)
  x_indices = max_indices // width
  x_indices = torch.tensor(x_indices, dtype=torch.float)
  y_indices = max_indices % width
  y_indices = torch.tensor(y_indices, dtype = torch.float)

  points = torch.stack((x_indices.squeeze(), y_indices.squeeze()), dim=-1) # reshpae the points into(-1, no.channels,2)
  if ( ( points.dim() ) > 2):

    normalized_tensor = torch.nn.functional.normalize(points, dim=-1) #normalize the points over the last dimension
    dot_products = torch.bmm(normalized_tensor, normalized_tensor.transpose(1,2)) #bmm is the same as matrix multiplication but for more than one matrix
    radiant_angles = torch.acos(dot_products) #calculate the angels in radiant

    radiant_angles = torch.nan_to_num(radiant_angles, nan=0) # Due to float points it might replace the zero with nan or 1.e-4 number.
    radiant_angles = torch.where(radiant_angles <= 0.0004, torch.tensor(0.0), radiant_angles)

    return radiant_angles

  else:
    # calculate radiant angles but for just a one batc, where the points.shape equal(no.channels,2)

    normalized_tensor = torch.nn.functional.normalize(points, dim=1) #normalize the tensor over the last dimension

    dot_products = torch.matmul(normalized_tensor, normalized_tensor.T)
    radiant_angles = torch.acos(dot_products)

    radiant_angles = torch.nan_to_num(radiant_angles, nan=0)  #Due to float points it might replace the zero with nan or 1.e-4 number.
    radiant_angles = torch.where(radiant_angles <= 0.0004, torch.tensor(0.0), radiant_angles)

    return radiant_angles


In [8]:
def LogEuclideanDifference(input_matrix, output_matrix):
  '''
  According to the loss function it should be log of the euclidean distance
  between the output and the ground truth, whether it's for e
   '''

  squared_diff = (input_matrix - output_matrix)**2

  sum_squared_diff = squared_diff.sum() #Sum all the squared differences element-wise

  euclidean_distance = torch.sqrt(sum_squared_diff) #Compute the square root to get the Euclidean distance

  log_difference = torch.log(euclidean_distance) / torch.log(torch.tensor(2.0)) # Log based two for euclidean_distance
  return log_difference.item()

In [9]:
def final_loss_function(predicted_output, ground_truth):
  '''
  Calling the functions that returns the Distances_matrix and the Angles matrix,
  for both ground truth and predicted output. then calculate the log of l2 distance
  between
  '''
  input_distances = GlobalMaxDistance(ground_truth)
  output_distances = GlobalMaxDistance(predicted_output)
  log_difference_distance_term = LogEuclideanDifference(input_distances, output_distances)


  ground_truth_radiant_angles = GlobalMaxAngle(ground_truth)
  predicted_output_angles = GlobalMaxAngle(predicted_output)
  log_difference_angel_term = LogEuclideanDifference(ground_truth_radiant_angles, predicted_output_angles)

  weight_term = log_difference_distance_term + log_difference_angel_term
  l2_loss = LossFunction(predicted_output, ground_truth)


  return (l2_loss)* (weight_term)


## UNET architecture

In [10]:
class UNet(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNet, self).__init__()

        # Encoding path
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

        # Decoding path
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2),  # Adjusted kernel size and stride
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2),  # Adjusted kernel size and stride
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, out_channels, kernel_size=4, stride=2),
            nn.MaxPool2d(kernel_size=4, stride=2) # Adjusted kernel size and stride
        )

    def forward(self, x):

        x1 = self.encoder(x) # Encoding path


        x2 = self.bottleneck(x1) # Bottleneck


        x3 = self.decoder(x2) # Decoding path

        return x3

## Testing the loss function on dummy data before training

In [11]:
model = UNet(in_channels=1, out_channels=19)
input = InputInit(2,512,512,1) # initiate dummy input
ground_truth = GroundTruthInit(2, 512, 512, 19) # initiate dummy output
output_data = model(input) #getting model's output
print("Output Shape:", output_data.shape)
loss = final_loss_function(output_data, ground_truth) #calculate the loss
print(f'\nThe loss is {loss}')

Output Shape: torch.Size([2, 19, 512, 512])

The loss is 5.552357196807861


<ipython-input-5-466e26052a92>:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x_indices = torch.tensor(x_indices, dtype=torch.float)
<ipython-input-5-466e26052a92>:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_indices = torch.tensor(y_indices, dtype = torch.float)
<ipython-input-6-737f8dea9e9b>:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x_indices = torch.tensor(x_indices, dtype=torch.float)
<ipython-input-6-737f8dea9e9b>:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach(

In [12]:
# Training and data loader preparation for batch training
batch_size = 2
learning_rate = 0.001
num_epochs = 5
num_batches = 2
train_input_data = torch.randn(batch_size*num_batches, 1, 512, 512)
train_target_data = torch.randn(batch_size*num_batches, 19, 512, 512)
train_dataset = TensorDataset(train_input_data, train_target_data)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
optimizer = optim.Adam(model.parameters(), lr=0.001) # optimize using ADAM

In [13]:
for epoch in range(num_epochs):
    for batch_idx, (inputs, targets) in enumerate(train_loader):

        optimizer.zero_grad() # Initiate the gradients


        outputs = model(inputs) # Forward pass
        loss = final_loss_function(outputs, targets) # Calculate the loss

        loss.backward() # Backpropagation


        optimizer.step() # Optimization for each step

        # monitoring batch loss
        print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx+1}/{num_batches}], Loss: {loss.item():.4f}")

<ipython-input-5-466e26052a92>:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x_indices = torch.tensor(x_indices, dtype=torch.float)
<ipython-input-5-466e26052a92>:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_indices = torch.tensor(y_indices, dtype = torch.float)
<ipython-input-6-737f8dea9e9b>:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x_indices = torch.tensor(x_indices, dtype=torch.float)
<ipython-input-6-737f8dea9e9b>:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach(

Epoch [1/5], Batch [1/2], Loss: 16.9980
Epoch [1/5], Batch [2/2], Loss: 16.5270
Epoch [2/5], Batch [1/2], Loss: 17.3023
Epoch [2/5], Batch [2/2], Loss: 17.5538
Epoch [3/5], Batch [1/2], Loss: 17.3092
Epoch [3/5], Batch [2/2], Loss: 17.2775
Epoch [4/5], Batch [1/2], Loss: 17.1537
Epoch [4/5], Batch [2/2], Loss: 17.5337
Epoch [5/5], Batch [1/2], Loss: 17.3838
Epoch [5/5], Batch [2/2], Loss: 17.4642


## Check if the loss function is differentiable

In [14]:
for name, param in model.named_parameters():
    if param.grad is not None:
        print(f"Parameter: {name}, Gradient Norm: {param.grad.norm()}")
    else:
        print(f"Parameter: {name}, Gradient: None")

Parameter: encoder.0.weight, Gradient Norm: 1.3327316082722973e-05
Parameter: encoder.0.bias, Gradient Norm: 1.0287234545103274e-05
Parameter: encoder.2.weight, Gradient Norm: 0.00019567328854463995
Parameter: encoder.2.bias, Gradient Norm: 3.0499533750116825e-05
Parameter: encoder.5.weight, Gradient Norm: 0.0005335473106242716
Parameter: encoder.5.bias, Gradient Norm: 0.0001298395509365946
Parameter: encoder.7.weight, Gradient Norm: 0.0003994488506577909
Parameter: encoder.7.bias, Gradient Norm: 0.0005615060799755156
Parameter: bottleneck.0.weight, Gradient Norm: 0.00012514402624219656
Parameter: bottleneck.0.bias, Gradient Norm: 0.00018456176621839404
Parameter: bottleneck.2.weight, Gradient Norm: 0.00018910728977061808
Parameter: bottleneck.2.bias, Gradient Norm: 0.00013207625306677073
Parameter: decoder.0.weight, Gradient Norm: 0.0007936481852084398
Parameter: decoder.0.bias, Gradient Norm: 0.0010375594720244408
Parameter: decoder.2.weight, Gradient Norm: 0.0010912015568464994
Para